## Autograd

In [1]:
import torch

In [2]:
if torch.cuda.is_available():
    print("CUDA is available. You can use GPU acceleration.")
else:
    print("CUDA is not available. You will be using CPU.")

CUDA is available. You can use GPU acceleration.


In [9]:

x = torch.tensor([1, 2, 3], dtype=torch.float32, requires_grad=True)

y = x**2
print(x)
print(y)

y.sum().backward()

print(x.grad)

tensor([1., 2., 3.], requires_grad=True)
tensor([1., 4., 9.], grad_fn=<PowBackward0>)
tensor([2., 4., 6.])


In [12]:
x = torch.tensor([1., 2., 3.], requires_grad=True)

y = x**2
z = torch.sin(y)

z.sum().backward()

print(x.grad)
print(type(x))
print(type(y))
print(type(z))


tensor([ 1.0806, -2.6146, -5.4668])
<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>


### cleaning grad 


In [73]:
# cleaning gradient
x=torch.tensor([1., 2., 3.], requires_grad=True)



In [74]:
y=(x**2).sum()

In [75]:
y.backward()
print("Gradient of y with respect to x:", x.grad)
# first time backward() is called, the gradient is computed and stored in x.grad
# from the second time backward() is called, the gradient will be accumulated in x.grad

Gradient of y with respect to x: tensor([2., 4., 6.])


In [76]:
# we need to clean the gradient before calling backward() again
x.grad.zero_()

tensor([0., 0., 0.])

In [77]:
# so the total code will be 
x=torch.tensor([1., 2., 3.], requires_grad=True)
y=(x**2).sum()

y.backward()
print("Gradient of y with respect to x:", x.grad)
x.grad.zero_()

Gradient of y with respect to x: tensor([2., 4., 6.])


tensor([0., 0., 0.])

## turning of backward pass

In [82]:
# after training , we dont need to compute the gradient, so we can use torch.no_grad() to save memory and computations
# we can just turn of the backward pass

#option 1
x.requires_grad_(False)
x
# here we cant call y.variable() because x.requires_grad is False, so we cant compute the gradient of y with respect to x

#option 2
z=x.detach()
z

#option 3
with torch.no_grad():
    y=x**2
    print(y)

tensor([1., 4., 9.])


## Lets's take a simplier perceptron

input (x) -- z=x*w+b --- y_pred= sigmoid(z) --- L=binary_cross_entropy(y_pred,y_true)

***manually calculating derivative***

In [34]:
x=torch.tensor(6.7)
y=torch.tensor(0.0)

w=torch.tensor(3.3)
b=torch.tensor(0.0)

In [35]:
def binary_cross_entropy(y_true, y_pred):
    epsilon = 1e-6  # Small value to avoid log(0)
    y_pred = torch.clamp(y_pred, epsilon, 1.0 - epsilon)  # Clamp predictions to avoid log(0)
    loss = - (y_true * torch.log(y_pred) + (1 - y_true) * torch.log(1 - y_pred))
    return loss

In [39]:
z=w*x+b
y_pred=torch.sigmoid(z)

epsilon = 1e-6
y_pred = torch.clamp(torch.sigmoid(z), epsilon, 1 - epsilon)

loss=binary_cross_entropy(y_true=y,y_pred=y_pred)

In [40]:
dloss_dy_pred = (y_pred - y) / (y_pred * (1 - y_pred))
dy_pred_dz = y_pred * (1 - y_pred)
dz_dw = x
dz_db = 1

dl_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dl_db = dloss_dy_pred * dy_pred_dz * dz_db

In [41]:
print("Gradient of loss with respect to w:", dl_dw)
print("Gradient of loss with respect to b:", dl_db)

Gradient of loss with respect to w: tensor(6.7000)
Gradient of loss with respect to b: tensor(1.0000)


***using autograd for derivatives***

In [44]:
x=torch.tensor(6.7)
y=torch.tensor(0.0)

w_autograd=torch.tensor(3.3, requires_grad=True)
b_autograd=torch.tensor(0.0, requires_grad=True)

In [45]:
z_autograd=w_autograd*x+b_autograd
y_pred_autograd=torch.sigmoid(z_autograd)

epsilon = 1e-6
y_pred_autograd = torch.clamp(torch.sigmoid(z_autograd), epsilon, 1 - epsilon)

loss=binary_cross_entropy(y_true=y,y_pred=y_pred_autograd)

In [47]:
z_autograd.backward()


In [48]:
dl_dw_autograd = w_autograd.grad
dl_db_autograd = b_autograd.grad

print("Gradient of loss with respect to w (autograd):", dl_dw_autograd)
print("Gradient of loss with respect to b (autograd):", dl_db_autograd)

Gradient of loss with respect to w (autograd): tensor(6.7000)
Gradient of loss with respect to b (autograd): tensor(1.)
